# Lottery Ticket Hypothesis

The core technique we are trying to test here is that of the Lottery Ticket Hypothesis, which posits that within a randomly initialized neural network, there exists a smaller sub-network (the "winning ticket") that, if reinitialized with the same initial weights, can train as effectively as the full network. In essence, the hypothesis suggests that large neural networks contain sparse, trainable sub-networks that, when identified and trained from the same starting point, can achieve comparable performance to the original network with fewer resources and training time.

In [1]:
import os

target_folder = "CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if not os.getcwd().endswith(target_folder):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/jack/college/scalableComputingForDataAnalytics/CS6423_knowledge_distillation_project/lottery_ticket_hypothesis/CS6423_knowledge_distillation_project'

In [3]:
import sys
sys.path.append('.')

from modules import ImagenetLoader, datasetPrepper, modelTrainer, ModelEvaluator
import pandas as pd

pretrained_weights_path = "RadImageNet_weights/resnet50.pth"
dataframe_path = "data/labels.csv"
image_dir = "data/test_images"
model_name = "radimagenet50_vqa_baseline1"

dataframe = pd.read_csv(dataframe_path)
num_classes = dataframe["label"].nunique()

dataframe.head()

ModuleNotFoundError: No module named 'modules'

In [ ]:
data = datasetPrepper(
    dataframe_path=dataframe_path,
    image_dir=image_dir,
).prepare(compute_class_weights=True)

print(f"Dataset prepared:")
print(f"  Train samples: {len(data.train_dataset)}")
print(f"  Val samples: {len(data.val_dataset)}")
print(f"  Classes: {len(data.class_names)}")

In [ ]:
loader = ImagenetLoader()
loader.load_radimagenet_resnet50(pretrained_weights_path)
loader.freeze_backbone()

model = loader.model
print(model)

In [ ]:
trainer = modelTrainer(
    model=model,
    data_prep=data,
    device=None,
    learn_rate=0.001,
    num_epochs=10,
    model_name=model_name,
)

trainer.prepare_for_training(trainable_params=loader.get_trainable_params())
print("Ready to train")

In [ ]:
trainer.train_all()

In [ ]:
pruning_percentages = [10, 25, 50, 75, 90]

In [ ]:
avg_errors_lth = []

for p_percent in pruning_percentages:
    print(f"\nRunning LTH for {p_percent}% pruning")

    trainer = modelTrainer(
        model=model,
        data_prep=data,
        device=None,
        learn_rate=0.001,
        num_epochs=10,
        model_name=model_name,
    )

    trainer.prepare_for_training(trainable_params=loader.get_trainable_params())
    print(f"Training parent model for {p_percent}% LTH pruning...")
    
    trainer.train_all()

    # b. Obtain the pruning mask from the parent model
    model.eval()
    masks = {}
    for name, module in model.named_modules():
      if isinstance(module, T.nn.Linear):
        # Temporarily apply pruning to get the mask, then remove for state_dict transfer
        prune.l1_unstructured(module, name='weight', amount=p_percent / 100.0)
        masks[name] = (module.weight_mask).cpu() # Store the binary mask from the reparameterized module
        prune.remove(module, 'weight') # Remove pruning reparameterization

    # c. Create and reinitialize the 'winning ticket' model
    lth_resnet = models.resnet50().to(device)
    lth_resnet.load_state_dict(pretrained_weights_path) # Reinitialize with original untrained weights

    # d. Apply the mask to the 'winning ticket' by directly setting weights to zero where mask is zero
    with T.no_grad():
      for name, module in lth_resnet.named_modules():
        if isinstance(module, T.nn.Linear) and name in masks:
            module.weight.data.mul_(masks[name].to(device))
            # To ensure proper pruning functionality during retraining, we can also apply permanent pruning here
            # This makes the zeroed weights truly zero and not trainable for the retraining phase.
            prune.custom_from_mask(module, name='weight', mask=masks[name].to(device))
            prune.remove(module, 'weight') # Make pruning permanent

    # e. Retrain the 'winning ticket'
    lth_trainer = modelTrainer(
        model=lth_resnet,
        data_prep=data,
        device=None,
        learn_rate=0.001,
        num_epochs=10,
        model_name=model_name,
    )
    lth_resnet.train()
    print(f"Retraining winning ticket for {p_percent}% LTH pruning...")
    lth_trainer.train_all()

    # f. Evaluate the 'winning ticket'
    lth_resnet.eval()
    err_list_lth = make_err_list(lth_resnet, data)
    current_avg_err_lth = np.mean([err for _, err in err_list_lth])
    avg_errors_lth.append(current_avg_err_lth)
    print(f"Average reconstruction error after {p_percent}% LTH pruning and retraining: {current_avg_err_lth:.4f}")

# Summarize findings for LTH
print("\n--- Lottery Ticket Hypothesis Experiment Summary ---")
print("The Lottery Ticket Hypothesis suggests that a randomly initialized, dense neural network contains a subnetwork that, when trained in isolation, can achieve comparable performance to the original network. In this experiment, for each pruning level, a 'parent' model was trained and pruned to get a mask. A new model was reinitialized with the original weights, the mask was applied, and then this 'winning ticket' was retrained. The average reconstruction error indicates how well these subnetworks perform compared to the fully trained models.")
print("------------------------------------------")

### Step 1: Initalise the model and save these initial weights

### Step 2: Little bit of fine tuning

### Step 3: Applying the submask to get the ticket

### Step 4: Reset the weights and train the ticket

### Step 5: Evaluate that ticket

## Applying all of this